# Split solver: a hand run on the demo fixture

**What this is.** Before `api/rules/allocation.py` is written (C4, Thu 15), the split model is run here on the
synthetic demo snapshot, to check that its rules give sensible answers. It's the "Assignment" in the
[design doc](../../docs/designs/resource-split-and-live-loop.md#the-assignment).

**The model** (design doc, "Split solver", with D33–D35 from 12 Oct):
- A resource needs the capability and access to the area, and has a capacity per help type.
- Boats and trucks count **seat units**, not heads (D33): adult 1, child under 12 half, baby on a lap 0,
  stretcher or wheelchair 3.
- A family goes whole by default. The solver may **suggest a split** only when that fits more people, and never
  leaves a child, a baby or a stretcher case without an adult, in the part that goes or the part that waits (D34).
  Every split needs the coordinator's approval.
- Accepted allocations stay fixed.
- The objective is lexicographic, solved level by level:
  1. **1a:** people covered whose need has a qualified urgent medical requirement;
  2. **1b:** other tier-1 people covered (coordinator urgent flag with a reason);
  3. **2:** tier-2 people covered, weighted `w_evac × evacuation + w_supply × supplies` (each weight ≥ 0.1, D35);
  4. **fewest split families**;
  5. **3:** least waiting left uncovered, in person-minutes (D35), as the last tie-break only.

Tiers come from the shared `api/rules/tiers.py` (C3), so the coordinator's split panel and the team lists rank
urgency the same way. The solver only *suggests*: it writes nothing, and teams still take work themselves.

**The checks:** (a) never more than 12 seat units on boat S1; (b) unmet need stays visible, with a reason;
(c) tier 1 is covered before tier 2; (d) the answer changes sensibly when S1 breaks down; plus the split
safeguards. All data is synthetic.

In [1]:
import json, sys
from collections import Counter
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from ortools.sat.python import cp_model
from api.rules.tiers import Tier, TierInput, colour, tier

SNAP = json.loads((ROOT / "contract/fixtures/allocationInput.cp3.json").read_text())
print(SNAP["label"])
print(SNAP["seatUnits"]["label"])

Synthetic snapshot for the split solver (C4). Demo, 10:25 Bangkok: T-S has accepted C-01's transport (M-01). Families carry their make-up (adults, children, babies, stretchers) for seat units (D33).
Seats a person takes on a boat or truck (D33, proposed): adult 1, child under 12 half, baby on a lap 0, stretcher or wheelchair 3. A team leader can override for their own vehicle.


## The snapshot

In [2]:
NOW = datetime.fromisoformat(SNAP["snapshotAt"].replace("Z", "+00:00"))
KINDS = ("adults", "children", "infants", "stretcher")

def half_seats(snap):
    return {k: int(2 * snap["seatUnits"][k]) for k in KINDS}  # integer half-seats for CP-SAT

def needs_of(snap):
    hs = half_seats(snap)
    out = []
    for doc in snap["needs"]:
        t = TierInput.from_listing(doc)
        party = doc.get("party")
        out.append({"id": doc["listingId"], "area": doc["areaId"], "type": doc["assistanceType"], "units": doc["units"],
                    "unit": doc["unit"], "party": party, "tier": tier(t), "colour": colour(t).value,
                    "seats": sum(party[k] * hs[k] for k in KINDS) / 2 if party else None,
                    "waiting_min": int((NOW - t.waiting_since).total_seconds() // 60), "note": doc.get("note", "")})
    return out

ONE = {"adults": "adult", "children": "child", "infants": "baby", "stretcher": "stretcher"}
MANY = {"adults": "adults", "children": "children", "infants": "babies", "stretcher": "stretchers"}

def who(p):
    return ", ".join(f"{p[k]} {(ONE if p[k] == 1 else MANY)[k]}" for k in KINDS if p.get(k))

print(f"{'need':<12}{'type':<22}{'units':>5}  seats  tier colour  waiting  family")
for n in needs_of(SNAP):
    seats = f"{n['seats']:>5}" if n["seats"] is not None else "    -"
    print(f"{n['id']:<12}{n['type']:<22}{n['units']:>5}  {seats}  {n['tier'].value:<4} {n['colour']:<7}{n['waiting_min']:>4} min  "
          + (who(n["party"]) if n["party"] else n["note"]))
print()
for r in SNAP["resources"]:
    print(f"{r['resourceId']} ({r['teamId']}, {r['kind']}): {r['capacity']} in {r['areas']} · {r['capacityBasis']}")

need        type                  units  seats  tier colour  waiting  family
L-C01-MED   medical                   1      -  1a   red      10 min  Qualified review: urgent medical (tier 1a)
L-C01-EVAC  evacuation_transport      3    2.5  1b   red      10 min  2 adults, 1 child
L-C02-EVAC  evacuation_transport      4    3.0  2    yellow   45 min  2 adults, 2 children
L-C03-EVAC  evacuation_transport      2    2.0  2    yellow  140 min  2 adults
L-C04-EVAC  evacuation_transport      5    3.0  2    yellow   75 min  2 adults, 2 children, 1 baby
L-C05-EVAC  evacuation_transport      3    5.0  2    yellow  155 min  2 adults, 1 stretcher
L-C06-EVAC  evacuation_transport      4    4.0  2    yellow   30 min  4 adults
L-C08-EVAC  evacuation_transport      3    2.0  2    yellow  115 min  1 adult, 2 children
L-C05-SUP   supplies                  3      -  2    green   155 min  No team with supplies can reach deep water
L-C12-MED   medical                   1      -  2    yellow   55 min  Non-urgen

## The model

In [3]:
TYPE_WEIGHT = {"evacuation_transport": "w_evac", "supplies": "w_supply", "medical": "w_evac"}
MIN_WEIGHT = 0.1  # D35: a zero weight could leave seats empty while people wait

def solve_split(snap, drop=(), extra=(), weights=None, allow_splits=True):
    """Suggest which resource covers which family (or safe part of one). Pure: reads the snapshot, writes nothing."""
    weights = {**snap["weights"], **(weights or {})}
    if min(weights.values()) < MIN_WEIGHT:
        raise ValueError(f"weights must be at least {MIN_WEIGHT}: {weights}")
    hs = half_seats(snap)
    resources = [r for r in snap["resources"] if r["available"] and r["resourceId"] not in drop] + list(extra)
    accepted = [a for a in snap["accepted"] if a["resourceId"] not in drop]  # a broken resource's jobs go back to the list
    needs = needs_of(snap)
    m = cp_model.CpModel()

    capable = {n["id"]: [r for r in resources if n["type"] in r["capabilities"] and n["area"] in r["areas"]] for n in needs}
    take, used, covered, splits = {}, {}, {}, []
    for n in needs:
        rs = capable[n["id"]]
        if n["party"]:  # people: count by kind, so a family can be split safely
            P, total = n["party"], n["units"]
            dependants = P["children"] + P["infants"] + P["stretcher"]
            for r in rs:
                k = (n["id"], r["resourceId"])
                take[k] = {kind: m.NewIntVar(0, P[kind], f"{kind}_{k}") for kind in KINDS}
                used[k] = m.NewBoolVar(f"used_{k}")
                part = sum(take[k].values())
                m.Add(part >= 1).OnlyEnforceIf(used[k])
                m.Add(part == 0).OnlyEnforceIf(used[k].Not())
                # D34: a part that goes with children, a baby or a stretcher case has an adult in it
                m.Add(take[k]["children"] + take[k]["infants"] + take[k]["stretcher"] <= dependants * take[k]["adults"])
            left = {kind: P[kind] - sum(take[n["id"], r["resourceId"]][kind] for r in rs) for kind in KINDS}
            for kind in KINDS:
                m.Add(left[kind] >= 0)
            # D34: ... and so does the part that waits
            m.Add(left["children"] + left["infants"] + left["stretcher"] <= dependants * left["adults"])
            covered[n["id"]] = total - sum(left.values())
            if rs:
                whole = m.NewBoolVar(f"whole_{n['id']}")
                none = m.NewBoolVar(f"none_{n['id']}")
                m.Add(covered[n["id"]] == total).OnlyEnforceIf(whole)
                m.Add(sum(used[n["id"], r["resourceId"]] for r in rs) <= 1).OnlyEnforceIf(whole)
                m.Add(covered[n["id"]] == 0).OnlyEnforceIf(none)
                split = m.NewBoolVar(f"split_{n['id']}")
                m.AddBoolOr([whole, none, split])  # anything else is a split
                if not allow_splits:
                    m.Add(split == 0)
                splits.append(split)
        else:  # medical visits and supply runs: whole or not at all
            for r in rs:
                used[n["id"], r["resourceId"]] = m.NewBoolVar(f"x_{n['id']}_{r['resourceId']}")
            covered[n["id"]] = n["units"] * sum(used[n["id"], r["resourceId"]] for r in rs) if rs else 0
            if rs:
                m.Add(sum(used[n["id"], r["resourceId"]] for r in rs) <= 1)

    for a in accepted:  # accepted allocations stay as they are (here: the whole family)
        n = next(x for x in needs if x["id"] == a["listingId"])
        for kind in KINDS:
            m.Add(take[a["listingId"], a["resourceId"]][kind] == n["party"][kind])

    for r in resources:  # capacity per help type; people in half-seats
        rid = r["resourceId"]
        for typ, cap in r["capacity"].items():
            if typ == "evacuation_transport":
                load = sum(sum(hs[kind] * v for kind, v in take[k].items()) for k in take if k[1] == rid)
                m.Add(load <= int(2 * cap))
            else:
                m.Add(sum(n["units"] * used[n["id"], rid] for n in needs
                          if (n["id"], rid) in used and n["type"] == typ) <= cap)

    def people(t):
        return sum(covered[n["id"]] for n in needs if n["tier"] == t)

    levels = [
        ("1a urgent medical people covered", people(Tier.T1A), "max"),
        ("1b other urgent people covered", people(Tier.T1B), "max"),
        ("2 weighted units covered (x100)", sum(int(round(100 * weights[TYPE_WEIGHT[n["type"]]])) * covered[n["id"]]
                                              for n in needs if n["tier"] == Tier.T2), "max"),
        ("split families", sum(splits), "min"),
        ("3 person-minutes left waiting", sum(n["waiting_min"] * (n["units"] - covered[n["id"]]) for n in needs), "min"),
    ]
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1  # deterministic: same answer every run
    values = {}
    for name, expr, sense in levels:  # lexicographic: fix each level before the next
        (m.Maximize if sense == "max" else m.Minimize)(expr)
        assert solver.Solve(m) == cp_model.OPTIMAL, name
        values[name] = int(round(solver.ObjectiveValue()))
        m.Add(expr >= values[name]) if sense == "max" else m.Add(expr <= values[name])

    by_id = {r["resourceId"]: r for r in resources}
    acc = {(a["listingId"], a["resourceId"]) for a in accepted}
    assignments, unassigned = [], []
    for n in needs:
        got = 0
        for r in capable[n["id"]]:
            k = (n["id"], r["resourceId"])
            if not solver.Value(used[k]):
                continue
            part = {kind: solver.Value(v) for kind, v in take[k].items()} if k in take else None
            units = sum(part.values()) if part else n["units"]
            got += units
            assignments.append({**n, "resource": r["resourceId"], "team": by_id[r["resourceId"]]["teamId"], "units": units,
                                "part": part, "accepted": k in acc})
        if got < n["units"]:
            reason = "capacity_exhausted" if capable[n["id"]] else "no_capable_team"
            left = ({kind: n["party"][kind] - sum(a["part"][kind] for a in assignments if a["id"] == n["id"]) for kind in KINDS}
                    if n["party"] else None)
            unassigned.append({**n, "units": n["units"] - got, "part": left, "reason": reason})
    rows = Counter(a["id"] for a in assignments)
    split_ids = {i for i, c in rows.items() if c > 1} | {u["id"] for u in unassigned if u["id"] in rows}
    for row in assignments + unassigned:
        row["split"] = row["id"] in split_ids
        row["needsCoordinatorApproval"] = row["split"]  # D34: the coordinator approves every split
    return {"inputVersions": snap["inputVersions"], "levels": values, "assignments": assignments,
            "unassigned": unassigned, "halfSeats": hs}


def label(n):
    return {"people": "person", "packs": "pack"}[n["unit"]] if n["units"] == 1 else n["unit"]

def seats(s, part):
    return sum(part[k] * s["halfSeats"][k] for k in KINDS) / 2

def show(s, title):
    print(f"== {title}")
    for name, v in s["levels"].items():
        print(f"   {name}: {v}")
    print("   Assigned:")
    for a in s["assignments"]:
        extra = f"  ({who(a['part'])}; {seats(s, a['part'])} seats)" if a["part"] else ""
        flags = ("  [accepted, fixed]" if a["accepted"] else "") + ("  [SPLIT: coordinator approves]" if a["split"] else "")
        print(f"     {a['id']:<12}{a['units']:>3} {label(a):<7}→ {a['resource']} ({a['team']})  tier {a['tier'].value}{extra}{flags}")
    print("   Not assigned (shown to the coordinator, never hidden):")
    for u in s["unassigned"]:
        extra = f"  ({who(u['part'])})" if u["part"] else ""
        print(f"     {u['id']:<12}{u['units']:>3} {label(u):<7}  tier {u['tier'].value}  {u['reason']}{extra}"
              + ("  [rest of a split family]" if u["split"] else ""))

def used_seats(s, rid):
    return sum(seats(s, a["part"]) for a in s["assignments"] if a["resource"] == rid and a["part"])

def evac_people(s):
    return sum(a["units"] for a in s["assignments"] if a["type"] == "evacuation_transport")

def safe(part):
    return part["adults"] >= 1 or not (part["children"] or part["infants"] or part["stretcher"])

## Run 1: the snapshot as it is (10:25)

In [4]:
base = solve_split(SNAP)
show(base, "Base: S1, R1 and M1 available")

== Base: S1, R1 and M1 available
   1a urgent medical people covered: 1
   1b other urgent people covered: 3
   2 weighted units covered (x100): 4300
   split families: 1
   3 person-minutes left waiting: 1150
   Assigned:
     L-C01-MED     1 person → M1 (T-M)  tier 1a
     L-C01-EVAC    3 people → S1 (T-S)  tier 1b  (2 adults, 1 child; 2.5 seats)  [accepted, fixed]
     L-C02-EVAC    3 people → S1 (T-S)  tier 2  (1 adult, 2 children; 2.0 seats)  [SPLIT: coordinator approves]
     L-C03-EVAC    2 people → S1 (T-S)  tier 2  (2 adults; 2.0 seats)
     L-C04-EVAC    5 people → S1 (T-S)  tier 2  (2 adults, 2 children, 1 baby; 3.0 seats)
     L-C08-EVAC    3 people → S1 (T-S)  tier 2  (1 adult, 2 children; 2.0 seats)
     L-PA-SUP     30 packs  → R1 (T-R)  tier 2
   Not assigned (shown to the coordinator, never hidden):
     L-C02-EVAC    1 person   tier 2  capacity_exhausted  (1 adult)  [rest of a split family]
     L-C05-EVAC    3 people   tier 2  capacity_exhausted  (2 adults, 1 stretch

**Reading it.** S1 has 12 seat units today and C-01's family (2 adults, 1 child = 2.5 seats) already has some of
them (accepted, fixed). Counting children as half a seat lets the boat take more people than 12 heads would. The
bedridden grandmother (C-05, a stretcher = 3 seats, plus 2 carers) is the hardest family to fit. M1 can make one
visit: C-01's urgent medical need (tier 1a) wins over C-12's routine check. R1 covers P-A's 30 packs but can't
enter deep water, so C-05's packs in P-B have no capable team: a real gap, which the split panel shows.

In [5]:
# (a) S1 never carries more than its 12 seat units
assert used_seats(base, "S1") <= 12, used_seats(base, "S1")

# (b) unmet need stays visible: covered + unassigned = all reviewed demand, each gap with a reason
unmet_evac = sum(u["units"] for u in base["unassigned"] if u["type"] == "evacuation_transport")
assert evac_people(base) + unmet_evac == 24
assert all(u["reason"] for u in base["unassigned"])

# (c) tier 1 before tier 2: C-01's medical need gets the only medical visit; C-01's transport is kept
got = {a["id"]: a["resource"] for a in base["assignments"]}
assert got["L-C01-MED"] == "M1" and got["L-C01-EVAC"] == "S1"
assert "L-C12-MED" in {u["id"] for u in base["unassigned"]}
assert not [u for u in base["unassigned"] if u["tier"] != Tier.T2]

# D34 safeguards: every part that goes, and every part that waits, is safe; every split needs approval
for row in base["assignments"] + base["unassigned"]:
    if row["part"]:
        assert safe(row["part"]), row
    assert row["needsCoordinatorApproval"] == row["split"]
print(f"S1 carries {evac_people(base)} people in {used_seats(base, 'S1')} of 12 seat units; {unmet_evac} people still waiting")
print("Checks (a), (b), (c) and the split safeguards pass")

S1 carries 16 people in 11.5 of 12 seat units; 8 people still waiting
Checks (a), (b), (c) and the split safeguards pass


## Seat units and splits: what they add

In [6]:
heads = json.loads(json.dumps(SNAP))
heads["seatUnits"].update({"children": 1.0, "infants": 1.0, "stretcher": 1.0})  # the old rule: one seat per head
by_heads = solve_split(heads, allow_splits=False)
whole_only = solve_split(SNAP, allow_splits=False)

print("People evacuated today on S1's 12 seats:")
print(f"  one seat per head, whole families only : {evac_people(by_heads)}")
print(f"  seat units,        whole families only : {evac_people(whole_only)}")
print(f"  seat units,        safe splits allowed : {evac_people(base)}")
assert evac_people(by_heads) <= evac_people(whole_only) <= evac_people(base)
# a split is only suggested when it fits more people
assert evac_people(base) > evac_people(whole_only) or not any(r["split"] for r in base["assignments"])

People evacuated today on S1's 12 seats:
  one seat per head, whole families only : 12
  seat units,        whole families only : 15
  seat units,        safe splits allowed : 16


**Finding: seat units can push the most vulnerable to the back.** The bedridden grandmother (C-05) has waited
longest of all, but she is left out: a stretcher takes 3 seats, so the boat fits more people without her. Counting
people is fair to families with children and unfair to stretcher cases. In real practice bedridden people are a
priority (Hat Yai's own officials kept a list of 230 bedridden patients). Proposal: the coordinator flags a
stretcher case urgent with a reason ("bedridden, can't self-evacuate"), which makes it tier 1b, so it goes before
any tier-2 family. The run below shows that.

In [7]:
flagged = json.loads(json.dumps(SNAP))
c05 = next(n for n in flagged["needs"] if n["listingId"] == "L-C05-EVAC")
c05["urgentFlag"] = {"by": "coordinator:demo-1", "reason": "Bedridden, can't self-evacuate"}
fix = solve_split(flagged)
show(fix, "C-05 flagged urgent by the coordinator (tier 1b)")
assert {"L-C05-EVAC"} <= {a["id"] for a in fix["assignments"]}
assert used_seats(fix, "S1") <= 12
print(f"The grandmother goes; S1 carries {evac_people(fix)} people")

== C-05 flagged urgent by the coordinator (tier 1b)
   1a urgent medical people covered: 1
   1b other urgent people covered: 6
   2 weighted units covered (x100): 3700
   split families: 1
   3 person-minutes left waiting: 1175
   Assigned:
     L-C01-MED     1 person → M1 (T-M)  tier 1a
     L-C01-EVAC    3 people → S1 (T-S)  tier 1b  (2 adults, 1 child; 2.5 seats)  [accepted, fixed]
     L-C04-EVAC    4 people → S1 (T-S)  tier 2  (1 adult, 2 children, 1 baby; 2.0 seats)  [SPLIT: coordinator approves]
     L-C05-EVAC    3 people → S1 (T-S)  tier 1b  (2 adults, 1 stretcher; 5.0 seats)
     L-C08-EVAC    3 people → S1 (T-S)  tier 2  (1 adult, 2 children; 2.0 seats)
     L-PA-SUP     30 packs  → R1 (T-R)  tier 2
   Not assigned (shown to the coordinator, never hidden):
     L-C02-EVAC    4 people   tier 2  capacity_exhausted  (2 adults, 2 children)
     L-C03-EVAC    2 people   tier 2  capacity_exhausted  (2 adults)
     L-C04-EVAC    1 person   tier 2  capacity_exhausted  (1 adult)  [r

**Note.** "One seat per head" counts a stretcher as 1 seat, so it can look generous; in reality it would overfill
the boat. Seat units are both more honest about stretchers and kinder to families with small children.

## Run 2: S1 breaks down (demo step 7, 10:30)

In [8]:
no_s1 = solve_split(SNAP, drop={"S1"})
show(no_s1, "S1 unavailable")

# (d) changes sensibly: nothing can reach P-B by water, so all 24 evacuees are unmet with a reason;
# C-01's medical visit (M-M) is NOT cancelled because the boat failed; P-A's packs are unchanged.
assert all(u["reason"] == "no_capable_team" for u in no_s1["unassigned"] if u["type"] == "evacuation_transport")
assert sum(u["units"] for u in no_s1["unassigned"] if u["type"] == "evacuation_transport") == 24
got = {a["id"]: a["resource"] for a in no_s1["assignments"]}
assert got["L-C01-MED"] == "M1" and got["L-PA-SUP"] == "R1"
print("Check (d) passes")

== S1 unavailable
   1a urgent medical people covered: 1
   1b other urgent people covered: 0
   2 weighted units covered (x100): 3000
   split families: 0
   3 person-minutes left waiting: 2315
   Assigned:
     L-C01-MED     1 person → M1 (T-M)  tier 1a
     L-PA-SUP     30 packs  → R1 (T-R)  tier 2
   Not assigned (shown to the coordinator, never hidden):
     L-C01-EVAC    3 people   tier 1b  no_capable_team  (2 adults, 1 child)
     L-C02-EVAC    4 people   tier 2  no_capable_team  (2 adults, 2 children)
     L-C03-EVAC    2 people   tier 2  no_capable_team  (2 adults)
     L-C04-EVAC    5 people   tier 2  no_capable_team  (2 adults, 2 children, 1 baby)
     L-C05-EVAC    3 people   tier 2  no_capable_team  (2 adults, 1 stretcher)
     L-C06-EVAC    4 people   tier 2  no_capable_team  (4 adults)
     L-C08-EVAC    3 people   tier 2  no_capable_team  (1 adult, 2 children)
     L-C05-SUP     3 packs    tier 2  no_capable_team
     L-C12-MED     1 person   tier 2  capacity_exhausted


**Reading it.** With S1 gone, its accepted job (M-01, C-01's transport) goes back to the list rather than staying
on a broken boat. That matches the demo: M-01 is held and relisted by the coordinator, the medical visit is
untouched, and P-A's plan doesn't move.

## Run 3: What if one more boat came? (the split panel's What-if)

In [9]:
S2 = {"resourceId": "S2", "teamId": "T-X", "kind": "boat", "capabilities": ["evacuation_transport"], "areas": ["P-B"],
      "capacity": {"evacuation_transport": 12}, "capacityBasis": "what-if: a second boat, 12 seat units", "available": True}
more = solve_split(SNAP, extra=[S2])
show(more, "What if: + boat S2 (simulation, not a real team)")
assert evac_people(more) == 24 and not any(r["split"] for r in more["assignments"])
assert used_seats(more, "S1") <= 12 and used_seats(more, "S2") <= 12
print("A second boat takes all 24 evacuees, with no family split")

== What if: + boat S2 (simulation, not a real team)
   1a urgent medical people covered: 1
   1b other urgent people covered: 3
   2 weighted units covered (x100): 5100
   split families: 0
   3 person-minutes left waiting: 520
   Assigned:
     L-C01-MED     1 person → M1 (T-M)  tier 1a
     L-C01-EVAC    3 people → S1 (T-S)  tier 1b  (2 adults, 1 child; 2.5 seats)  [accepted, fixed]
     L-C02-EVAC    4 people → S2 (T-X)  tier 2  (2 adults, 2 children; 3.0 seats)
     L-C03-EVAC    2 people → S2 (T-X)  tier 2  (2 adults; 2.0 seats)
     L-C04-EVAC    5 people → S2 (T-X)  tier 2  (2 adults, 2 children, 1 baby; 3.0 seats)
     L-C05-EVAC    3 people → S1 (T-S)  tier 2  (2 adults, 1 stretcher; 5.0 seats)
     L-C06-EVAC    4 people → S1 (T-S)  tier 2  (4 adults; 4.0 seats)
     L-C08-EVAC    3 people → S2 (T-X)  tier 2  (1 adult, 2 children; 2.0 seats)
     L-PA-SUP     30 packs  → R1 (T-R)  tier 2
   Not assigned (shown to the coordinator, never hidden):
     L-C05-SUP     3 packs    t

## Weights only move tier 2, and never go below 0.1 (D35)

In [10]:
try:
    solve_split(SNAP, weights={"w_evac": 0.0})
    raise AssertionError("a zero weight should be refused")
except ValueError as e:
    print("Refused:", e)

low = solve_split(SNAP, weights={"w_evac": 0.1})
got = {a["id"]: a["resource"] for a in low["assignments"]}
assert got["L-C01-MED"] == "M1" and got["L-C01-EVAC"] == "S1"
assert evac_people(low) == evac_people(base)  # at the lowest weight the boat is still filled
print(f"w_evac = 0.1: tier 1 unchanged, S1 still carries {evac_people(low)} people")

Refused: weights must be at least 0.1: {'w_evac': 0.0, 'w_supply': 1.0}
w_evac = 0.1: tier 1 unchanged, S1 still carries 16 people


## What this shows, and what's next

- The rules hold on the demo: S1 is never overfilled, the people without a seat stay visible, C-01 comes first,
  and the S1 failure gives the answer the demo script expects.
- Seat units (D33) and safe splits (D34) fit more people on the same boat, without leaving a child, a baby or a
  stretcher case without an adult. Every split is a suggestion for the coordinator to approve.
- Seat units alone would leave the bedridden grandmother waiting; a coordinator urgent flag (tier 1b) puts her
  first. Proposal: the desk prompts for that flag whenever a request includes a stretcher or wheelchair.
- Weights below 0.1 are refused (D35), so seats are always filled first and waiting only breaks ties. Waiting
  counts per person; tier-2 medical needs use `w_evac`.
- Still to confirm with the friend: D33–D35, and that the intake records adults, children, babies and stretchers.
- Next (Thu 15): move `solve_split` into `api/rules/allocation.py` as a pure function `solve_split(snapshot) -> suggestion`,
  with per-need explanation fields for Gemini and a time limit, and add the solver half of T15 to the tests.